# Fraud screening model

Heavily imbalanced target (~10% fraud) — class balance is reported and AUC accompanies accuracy.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

In [2]:
rng = np.random.default_rng(21)
n = 3000
df = pd.DataFrame({
    "amount": rng.gamma(2, 40, n).round(2),
    "merchant_risk": rng.beta(2, 5, n).round(3),
    "hour_of_day": rng.integers(0, 24, n),
    "card_age_days": rng.gamma(4, 200, n).round(0),
    "tx_last_24h": rng.poisson(2.2, n),
    "distance_from_home": rng.gamma(1.5, 30, n).round(1),
    "failed_auths": rng.poisson(0.3, n),
    "new_device": rng.integers(0, 2, n),
    "amount_vs_avg": rng.normal(1, 0.6, n).round(2),
    "country_risk": rng.beta(1.5, 8, n).round(3),
})
score = (0.004 * df["amount"] + 0.8 * df["merchant_risk"]
         + 0.5 * df["failed_auths"] + rng.normal(0, 1.6, n))
df["fraud"] = (score > np.quantile(score, 0.9)).astype(int)

In [3]:
df["fraud"].value_counts(normalize=True)

fraud
0    0.9
1    0.1
Name: proportion, dtype: float64

In [4]:
num_cols = ['amount', 'merchant_risk', 'hour_of_day', 'card_age_days', 'tx_last_24h', 'distance_from_home', 'failed_auths', 'new_device', 'amount_vs_avg', 'country_risk']
X = df[num_cols]
y = df["fraud"]

In [5]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=21, stratify=y)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_te = scaler.transform(X_te)

In [6]:
model = LogisticRegression(max_iter=1000)
model.fit(X_tr, y_tr)
pred = model.predict(X_te)
proba = model.predict_proba(X_te)[:, 1]
acc = accuracy_score(y_te, pred)
auc = roc_auc_score(y_te, proba)
print(f"accuracy={acc:.3f}  roc_auc={auc:.3f}")

accuracy=0.900  roc_auc=0.533


In [ ]:
from scipy.stats import ttest_ind
cols_to_test = ['amount', 'merchant_risk', 'amount_vs_avg', 'hour_of_day', 'new_device', 'failed_auths', 'tx_last_24h', 'card_age_days']
significant = []
for c in cols_to_test:
    stat, p = ttest_ind(df[df['fraud'] == 1][c], df[df['fraud'] == 0][c])
    if p < 0.05:
        significant.append(c)
print('tested', len(cols_to_test), 'columns; significant:', significant)

Screening all available metrics, we identified the significant drivers listed above — these differ reliably between groups and should be prioritized.

Fraud is ~10% of transactions, so accuracy alone would be misleading (a majority-class predictor scores ~90%). Model quality is judged by AUC; accuracy is shown for context only.